In [1]:
#Window function normal aggregate  se alag hai- aggregate (GROUP BY) rows ko ek single row me compress kar deta hai,
#lekin window function har row ko indivisually rakhta hai or uske sath ek "calculated column" add kar deta hai.
#yeh bahut powerful hai jab hume row-level detail + group-level context dono chaiye ho.

In [2]:
#01:Bsic OVER() - running total(cumulative sum):

In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    SELECT Invoice, "Customer ID", InvoiceDate, OrderValue,
            SUM(OrderValue) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate) as running_total
    FROM orders
    WHERE "Customer ID" = (SELECT "Customer ID" FROM orders LIMIT 1)
    ORDER BY InvoiceDate
""",conn)
print(q1)

   Invoice  Customer ID          InvoiceDate  OrderValue  running_total
0   489434      13085.0  2009-12-01 07:45:00      505.30         505.30
1   489435      13085.0  2009-12-01 07:46:00      145.80         651.10
2   490068      13085.0  2009-12-03 14:06:00      284.30         935.40
3   490069      13085.0  2009-12-03 14:07:00      161.40        1096.80
4   496092      13085.0  2010-01-29 10:06:00      430.20        1527.00
5   496166      13085.0  2010-01-29 11:42:00      490.20        2017.20
6  C527339      13085.0  2010-10-15 15:35:00     -830.12        1187.08
7   544306      13085.0  2011-02-17 13:57:00      278.10        1465.18
8  C551464      13085.0  2011-04-28 16:15:00     -143.70        1321.48
9   558996      13085.0  2011-07-05 12:11:00      137.98        1459.46


In [4]:
#PARTITION BY = GROUP BY jaisa hi hai lekin rows collapse nahi hoti; ORDER BY (OVER ke andar) yeh decide karta hai ki running calculation kis order me ho.

In [5]:
#02: AVG()OVER - Comparision ke liye (har row ka apne group ke average se fark dikhana):

In [8]:
q2 = pd.read_sql("""
    SELECT o. "Customer ID", Country, OrderValue,
           AVG(OrderValue) OVER (PARTITION BY Country) as country_avg_order,
           OrderValue - AVG(OrderValue) OVER (PARTITION BY Country) as diff_from_avg
    FROM orders o
    INNER JOIN customers c ON o."Customer ID" = c."Customer ID"
    ORDER BY Country
    LIMIT 20
""", conn)
print(q2)

    Customer ID    Country  OrderValue  country_avg_order  diff_from_avg
0       16321.0  Australia      196.10        1285.307463   -1089.207463
1       12422.0  Australia       75.00        1285.307463   -1210.307463
2       12431.0  Australia     1075.27        1285.307463    -210.037463
3       12431.0  Australia       45.00        1285.307463   -1240.307463
4       12422.0  Australia      662.25        1285.307463    -623.057463
5       12416.0  Australia      202.56        1285.307463   -1082.747463
6       12389.0  Australia      164.85        1285.307463   -1120.457463
7       12431.0  Australia      394.59        1285.307463    -890.717463
8       16321.0  Australia       34.80        1285.307463   -1250.507463
9       12422.0  Australia      396.20        1285.307463    -889.107463
10      12392.0  Australia      234.75        1285.307463   -1050.557463
11      12422.0  Australia      662.25        1285.307463    -623.057463
12      12422.0  Australia      396.20        1285.

In [9]:
#03: COUNT()OVER - har row ke sath uske group ka total count nikalna:

In [11]:
q3 = pd.read_sql("""
    SELECT o. "Customer ID", Country, Invoice, OrderValue,
           COUNT(*) OVER (PARTITION BY Country) as total_orders_in_country
    FROM orders o
    INNER JOIN customers c ON o."Customer ID" = c."Customer ID"
    LIMIT 20
""", conn)
print(q3)

    Customer ID    Country Invoice  OrderValue  total_orders_in_country
0       16321.0  Australia  489450      196.10                      134
1       12422.0  Australia  492744       75.00                      134
2       12431.0  Australia  494511     1075.27                      134
3       12431.0  Australia  494513       45.00                      134
4       12422.0  Australia  497879      662.25                      134
5       12416.0  Australia  498550      202.56                      134
6       12389.0  Australia  498617      164.85                      134
7       12431.0  Australia  500008      394.59                      134
8       16321.0  Australia  502275       34.80                      134
9       12422.0  Australia  503860      396.20                      134
10      12392.0  Australia  506115      234.75                      134
11      12422.0  Australia  507060      662.25                      134
12      12422.0  Australia  507061      396.20                  

In [12]:
#04difference samjhne ke liye side by side- GROUP BY vs Window function.

In [14]:
# GROUP BY version — rows collapse ho jaati hain
q4a = pd.read_sql("""
    SELECT Country, AVG(OrderValue) as avg_order
    FROM orders o
    INNER JOIN customers c ON o."Customer ID" = c."Customer ID"
    GROUP BY Country
""", conn)
print("GROUP BY result shape:", q4a.shape)

# Window Function version — sab rows preserve hoti hain
q4b = pd.read_sql("""
    SELECT o. "Customer ID", Country, OrderValue,
           AVG(OrderValue) OVER (PARTITION BY Country) as avg_order
    FROM orders o
    INNER JOIN customers c ON o."Customer ID" = c."Customer ID"
""", conn)
print("Window Function result shape:", q4b.shape)

GROUP BY result shape: (41, 2)
Window Function result shape: (45028, 4)


In [15]:
#ye dikhata hai ki GROUP BY sirf unique countries jitni rows deta hai, jabki window function poori original row count nikalta hai.

In [16]:
#practice questions.

In [17]:
#01:Har customer ke orders ka running total nikaalo (PARTITION BY "Customer ID"), lekin is baar top 5 customers ke liye (by total spend).

In [18]:
#

In [19]:
#02:Har country ka MAX(OrderValue) nikaalo Window Function se, aur ek naya column banao jo dikhaye ki kaunsa order us country ka sabse bada order hai (OrderValue = MAX(...) OVER(...)).

In [20]:
#

In [21]:
conn.close()